In [1]:
"""
Realtime Squat Counter & Quality Feedback
Final Year Project - Stable Version
"""

import cv2
import mediapipe as mp
import numpy as np
import time
import pandas as pd
from collections import deque
import pyttsx3   # for voice feedback
import joblib
import os

# ---------------- CONFIG ----------------
MODEL_PATH = "../data/squat_model.pkl"   # Trained model
OUTPUT_LOG_DIR = "../data"               # Save session logs
FEATURE_NAMES = [
    "knee_angle_r", "knee_angle_l",
    "elbow_angle_r", "elbow_angle_l",
    "hip_angle_r", "hip_angle_l",
    "shoulder_angle_r", "shoulder_angle_l"
]

DOWN_THRESHOLD = 90
UP_THRESHOLD = 170
SMOOTHING_WINDOW = 7
HOLD_TIME = 0.4
MIN_VISIBILITY = 0.4
FPS_LIMIT = 0.02   # ~50 FPS
# -----------------------------------------

# Load model if available
model = None
if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    print("✅ Model loaded")
else:
    print("⚠️ Model not found, fallback to rule-based feedback only")

# Voice engine
engine = pyttsx3.init()
engine.setProperty("rate", 170)

# Mediapipe setup
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(min_detection_confidence=0.6, min_tracking_confidence=0.6)

# --- Helpers ---
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

def speak(text):
    engine.say(text)
    engine.runAndWait()

# --- State vars ---
rep_count = 0
state = "up"
last_state_change = time.time()
knee_buffer = deque(maxlen=SMOOTHING_WINDOW)
rep_log = []
last_feedback_time = 0

# Camera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("❌ Camera not available")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            # visibility check
            vis_ids = [23,24,25,26,27,28]
            if np.mean([lm[i].visibility for i in vis_ids]) < MIN_VISIBILITY:
                cv2.putText(frame, "Low visibility - adjust camera",
                            (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,160,255), 2)
                cv2.imshow("Squat AI", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                continue

            # key joints
            shoulder_r, elbow_r, wrist_r = (lm[12].x, lm[12].y), (lm[14].x, lm[14].y), (lm[16].x, lm[16].y)
            hip_r, knee_r, ankle_r = (lm[24].x, lm[24].y), (lm[26].x, lm[26].y), (lm[28].x, lm[28].y)
            shoulder_l, elbow_l, wrist_l = (lm[11].x, lm[11].y), (lm[13].x, lm[13].y), (lm[15].x, lm[15].y)
            hip_l, knee_l, ankle_l = (lm[23].x, lm[23].y), (lm[25].x, lm[25].y), (lm[27].x, lm[27].y)

            # angles
            knee_angle_r = calculate_angle(hip_r, knee_r, ankle_r)
            knee_angle_l = calculate_angle(hip_l, knee_l, ankle_l)
            elbow_angle_r = calculate_angle(shoulder_r, elbow_r, wrist_r)
            elbow_angle_l = calculate_angle(shoulder_l, elbow_l, wrist_l)
            hip_angle_r = calculate_angle(shoulder_r, hip_r, knee_r)
            hip_angle_l = calculate_angle(shoulder_l, hip_l, knee_l)
            shoulder_angle_r = calculate_angle(elbow_r, shoulder_r, hip_r)
            shoulder_angle_l = calculate_angle(elbow_l, shoulder_l, hip_l)

            # smoothing
            avg_knee = (knee_angle_r + knee_angle_l) / 2
            knee_buffer.append(avg_knee)
            smooth_knee = float(np.median(knee_buffer))

            # quality prediction
            quality = "GOOD"
            if model:
                features = pd.DataFrame([[
                    knee_angle_r, knee_angle_l,
                    elbow_angle_r, elbow_angle_l,
                    hip_angle_r, hip_angle_l,
                    shoulder_angle_r, shoulder_angle_l
                ]], columns=FEATURE_NAMES)
                pred = model.predict(features)[0]
                quality = "GOOD" if int(pred) == 1 else "BAD"

            # rep detection
            now = time.time()
            if state == "up" and smooth_knee < DOWN_THRESHOLD:
                if now - last_state_change > HOLD_TIME:
                    state = "down"
                    last_state_change = now
            elif state == "down" and smooth_knee > UP_THRESHOLD:
                if now - last_state_change > HOLD_TIME:
                    state = "up"
                    rep_count += 1
                    last_state_change = now
                    rep_log.append({
                        "time": now,
                        "rep": rep_count,
                        "quality": quality,
                        "knee_angle": smooth_knee
                    })
                    # voice only every few seconds
                    if quality == "GOOD":
                        speak("Good rep")
                    else:
                        speak("Bad squat, keep your back straight")

            # draw pose
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # feedback box (top-right corner)
            overlay = frame.copy()
            cv2.rectangle(overlay, (frame.shape[1]-260, 20), (frame.shape[1]-20, 160), (50,50,50), -1)
            frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

            cv2.putText(frame, f"Reps: {rep_count}", (frame.shape[1]-240, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,0), 2)
            cv2.putText(frame, f"Quality: {quality}", (frame.shape[1]-240, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0) if quality=="GOOD" else (0,0,255), 2)
            cv2.putText(frame, f"State: {state}", (frame.shape[1]-240, 140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200,200,200), 2)

        cv2.imshow("Squat AI", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        time.sleep(FPS_LIMIT)

finally:
    cap.release()
    cv2.destroyAllWindows()
    if rep_log:
        out_path = os.path.join(OUTPUT_LOG_DIR, f"squat_log_{int(time.time())}.csv")
        pd.DataFrame(rep_log).to_csv(out_path, index=False)
        print(f"✅ Log saved: {out_path}")
    print(f"✅ Session ended. Total reps: {rep_count}")

✅ Model loaded


I0000 00:00:1757878432.453087 4570372 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1757878432.550859 4573082 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757878432.575492 4573082 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757878434.175648 4573087 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


✅ Log saved: ../data/squat_log_1757878541.csv
✅ Session ended. Total reps: 5
